In [6]:
import pandas as pd
import joblib

In [7]:
# Crop
crop_model = joblib.load("crop_model.pkl")
crop_encoders = joblib.load("encoders.pkl")
crop_features = joblib.load("crop_features.pkl")

# Yield
yield_model = joblib.load("yield_model.pkl")
yield_encoders = joblib.load("yield_encoders.pkl")
yield_features = joblib.load("yield_features.pkl")

# Price
price_model = joblib.load("price_model.pkl")
price_encoders = joblib.load("price_encoders.pkl")
price_features = joblib.load("price_features.pkl")

In [11]:
import pandas as pd
import joblib

# Load everything once
model = joblib.load("crop_model.pkl")
encoders = joblib.load("encoders.pkl")
features = joblib.load("crop_features.pkl")

def predict_crop_final(N, Soil_pH, Temperature, Rainfall, Region="North"):
    
    # Step 1: User input (minimal)
    user_data = {
        'N': N,
        'Soil_pH': Soil_pH,
        'Temperature': Temperature,
        'Rainfall': Rainfall,
        'Region': Region
    }

    # Step 2: Smart defaults (based on your dataset)
    user_data.update({
        'P': 40,
        'K': 40,
        'Soil_Moisture': 40,
        'Soil_Type': 'Loamy',
        'Organic_Carbon': 0.8,
        'Electrical_Conductivity': 1.0,
        'Humidity': 60,
        'Sunlight_Hours': 8,
        'Wind_Speed': 5,
        'Altitude': 200,
        'Season': 'Kharif',
        'Irrigation_Type': 'Drip',
        'Fertilizer_Used': 100,
        'Previous_Crop': 'Wheat'
    })

    # Step 3: Convert to DataFrame
    df_input = pd.DataFrame([user_data])

    # Step 4: Encode categorical safely
    for col in encoders:
        if col in df_input:
            df_input[col] = encoders[col].transform(df_input[col])

    # Step 5: Ensure correct order
    df_input = df_input.reindex(columns=features)

    # Step 6: Predict
    prediction = model.predict(df_input)[0]

    # Step 7: Decode output
    crop_name = encoders["Recommended_Crop"].inverse_transform([prediction])[0]

    return crop_name

In [12]:
def predict_yield_final(crop, temperature, rainfall, state, district, year=2020):

    input_data = {
        'Crop': crop,
        'Crop_Year': year,
        'Season': 'Kharif',
        'State_Name': state,
        'District_Name': district
    }

    df_input = pd.DataFrame([input_data])

    # Safe encoding
    for col in yield_encoders:
        if col in df_input:
            if df_input[col][0] in yield_encoders[col].classes_:
                df_input[col] = yield_encoders[col].transform(df_input[col])
            else:
                df_input[col] = 0

    df_input = df_input.reindex(columns=yield_features)

    predicted_yield = yield_model.predict(df_input)[0]

    return predicted_yield

In [13]:
def predict_price_final(crop, state, district):

    input_data = {
        'State': state,
        'District': district,
        'Market': 'Default',   # auto
        'Commodity': crop,
        'Variety': 'Other',    # auto
        'Grade': 'FAQ',        # auto
        'Year': 2023,          # auto
        'Month': 7             # auto
    }

    df_input = pd.DataFrame([input_data])

    # Encode safely
    for col in price_encoders:
        if col in df_input:
            if df_input[col][0] in price_encoders[col].classes_:
                df_input[col] = price_encoders[col].transform(df_input[col])
            else:
                df_input[col] = 0

    df_input = df_input.reindex(columns=price_features)

    predicted_price = price_model.predict(df_input)[0]

    return predicted_price

In [14]:
def get_export_details_ai(crop, local_price):

    crop_category = {
        "Rice": "Staple",
        "Wheat": "Staple",
        "Maize": "Staple",
        "Banana": "Fruit",
        "Mango": "Fruit",
        "Potato": "Vegetable",
        "Onion": "Vegetable",
        "Cotton": "Cash Crop",
        "Sugarcane": "Cash Crop",
        "Moringa": "Superfood",
        "Turmeric": "Spice"
    }

    category_rules = {
        "Staple": [("UAE", 2.0), ("Bangladesh", 1.8), ("Saudi Arabia", 2.2)],
        "Fruit": [("UAE", 2.5), ("Netherlands", 3.0), ("UK", 2.8)],
        "Vegetable": [("Bangladesh", 1.6), ("Nepal", 1.5), ("Sri Lanka", 1.7)],
        "Cash Crop": [("China", 2.2), ("Vietnam", 2.0), ("Turkey", 2.3)],
        "Spice": [("USA", 4.0), ("Germany", 3.8), ("UK", 3.5)],
        "Superfood": [("USA", 8.0), ("Canada", 7.5), ("Australia", 7.0)]
    }

    # Step 1: Category
    category = crop_category.get(crop, "Staple")

    # Step 2: Get options
    options = category_rules.get(category, [("UAE", 2.0)])

    # Step 3: Calculate export prices
    results = []
    for country, multiplier in options:
        export_price = local_price * multiplier
        results.append((country, export_price))

    # Step 4: Sort (AI-like ranking)
    results = sorted(results, key=lambda x: x[1], reverse=True)

    return results, category

In [26]:
def smart_farming_system(N, Soil_pH, Temperature, Rainfall, state, district):

    crop = predict_crop_final(N, Soil_pH, Temperature, Rainfall)

    yield_val = predict_yield_final(
        crop=crop,
        temperature=Temperature,
        rainfall=Rainfall,
        state=state,
        district=district
    )

    local_price = predict_price_final(
        crop=crop,
        state=state,
        district=district
    )

    export_results, category = get_export_details_ai(crop, local_price)
    best_country, export_price = export_results[0]

    # Demand
    if export_price > local_price * 5:
        demand = "High"
    elif export_price > local_price * 2:
        demand = "Medium"
    else:
        demand = "Low"

    # Profit
    local_revenue = yield_val * local_price
    export_revenue = yield_val * export_price
    profit_gain = export_revenue - local_revenue
    profit_percent = (profit_gain / local_revenue) * 100 if local_revenue != 0 else 0

    # Decision
    if export_revenue > local_revenue:
        decision = "EXPORT MARKET"
    else:
        decision = "LOCAL MARKET"
    return {
       "crop": crop,
       "category": category,
       "state": state,
       "yield": round(float(yield_val), 2),
       "local_price": round(float(local_price), 2),
       "export_price": round(float(export_price), 2),
       "country": best_country,
       "demand": demand,
       "profit_percent": round(float(profit_percent), 2),
       "local_revenue": round(float(local_revenue), 0),
       "export_revenue": round(float(export_revenue), 0),
       "profit_gain": round(float(profit_gain), 0),
       "decision": decision
}

In [30]:
def display_result(result):

    print("\nSMART FARMING & EXPORT OPTIMIZATION SYSTEM")
    print("==========================================\n")

    print(f"Crop Recommendation       : {result['crop']}")
    print(f"Category                  : {result['category']}")
    print(f"Suitable Region           : {result['state']} (Based on {result['country']} export)")

    print(f"Predicted Yield (tons)    : {result['yield']:.2f}\n")

    print(f"Local Market Price (₹)    : {result['local_price']:.2f}/kg")
    print(f"Export Market Price (₹)   : {result['export_price']:.2f}/kg")
    print(f"Best Export Country       : {result['country']}\n")

    print(f"Global Demand Level       : {result['demand']}")
    print(f"Profit Margin (%)         : {result['profit_percent']:.2f}%\n")

    print("------------------------------------------")
    print("PROFIT ANALYSIS")
    print("------------------------------------------")

    print(f"Local Market Revenue (₹)  : {int(result['local_revenue']):,}")
    print(f"Export Market Revenue (₹) : {int(result['export_revenue']):,}")

    print(f"\nEstimated Profit Gain     : +{int(result['profit_gain']):,}")

    print("\n------------------------------------------")
    print("RECOMMENDATION")
    print("------------------------------------------")

    print(f"Better Option: {result['decision']} ✅\n")

    print("Reason:")

    if result['decision'] == "EXPORT MARKET":
        print("- Higher export price compared to local market")
        print("- Better profit margin")
        print("- Strong international demand")
    else:
        print("- Stable local market")
        print("- Lower risk")
        print("- Consistent pricing")

    print("==========================================")

In [31]:
result = smart_farming_system(
    N=90,
    Soil_pH=6.5,
    Temperature=25,
    Rainfall=1000,
    state="Tamil Nadu",
    district="Coimbatore"
)

display_result(result)


SMART FARMING & EXPORT OPTIMIZATION SYSTEM

Crop Recommendation       : Potato
Category                  : Vegetable
Suitable Region           : Tamil Nadu (Based on Sri Lanka export)
Predicted Yield (tons)    : 1237.91

Local Market Price (₹)    : 985.67/kg
Export Market Price (₹)   : 1675.63/kg
Best Export Country       : Sri Lanka

Global Demand Level       : Low
Profit Margin (%)         : 70.00%

------------------------------------------
PROFIT ANALYSIS
------------------------------------------
Local Market Revenue (₹)  : 1,220,168
Export Market Revenue (₹) : 2,074,285

Estimated Profit Gain     : +854,117

------------------------------------------
RECOMMENDATION
------------------------------------------
Better Option: EXPORT MARKET ✅

Reason:
- Higher export price compared to local market
- Better profit margin
- Strong international demand
